# AL2002: Artificial Intelligence — Lab 5

| | |
|---|---|
| **Name** | Zohaib Hussain |
| **Roll Number** | 23L-3087 |
| **Date** | 14 March 2026 |

---
## Task 1: Alpha–Beta Pruning

### 1A — Implementation

Implement `alpha_beta(node, depth, alpha, beta, is_max_player)` that:
- Accepts a nested list as the game tree (leaves are integers).
- Uses `depth` as a countdown: decrement at each recursive call. However, for this tree the **leaf check** (`isinstance(node, int)`) will trigger before depth reaches 0 — so the primary termination condition is the leaf check.
- Correctly updates alpha at MAX nodes and beta at MIN nodes.
- Prunes branches when `alpha >= beta`.
- Increments the global `nodes_evaluated` counter for **every node visited** (both internal nodes and leaves).
- Returns the optimal minimax value.

In [12]:
nodes_evaluated = 0  # increment this for every node visited inside alpha_beta

def is_leaf_node(node , depth):
    return depth == 0 or not isinstance(node, list)


def alpha_beta(node, depth, alpha, beta, is_max_player):
    """
    Performs Alpha-Beta Pruning on a game tree.

    Args:
        node            : current node — either a list (internal) or int (leaf)
        depth           : remaining depth to search
        alpha           : best value MAX can guarantee so far
        beta            : best value MIN can guarantee so far
        is_max_player   : True if current player is MAX, False if MIN

    Returns:
        int: optimal minimax value for this node
    """
    global nodes_evaluated 
    nodes_evaluated += 1

    # leaf node reached
    if(is_leaf_node(node, depth)):
        return node

    # travesing on all child nodes
    for child in node:

        # prune the branch
        if(is_max_player and alpha >= beta):
            return alpha
        elif(not is_max_player and beta <= alpha):
            return beta
            
        value = alpha_beta(child, depth - 1, alpha, beta, not is_max_player)
        if is_max_player:
            alpha = max(value, alpha)
        else:
            beta = min(value, beta)
      
        

    if is_max_player:
        return alpha
    else:
        return beta  

### 1B — Run and report

Call your function on the test tree below using `depth=4` and `is_max_player=True`.  
Print the **optimal value** and the **total nodes evaluated**.

In [17]:
nodes_evaluated = 0

game_tree = [
    [[[3, 17], [2, 12]], [[15,  2], [6,  1]]],
    [[[5,  3], [8,  9]], [[ 7, 12], [4, 11]]]
]
# Expected optimal value (MAX player at root): 6
# Expected nodes_evaluated (counting all visited nodes, including leaves): ~21
# (Exact count may vary slightly depending on how you count the root,
#  but with proper pruning it should be well below the full 31 nodes.)

# YOUR CODE HERE
result = alpha_beta(game_tree, depth=4, alpha=float('-inf'), beta=float('inf'), is_max_player=True)
print(f"Optimal value : {result}")
print(f"Nodes evaluated: {nodes_evaluated}")

Optimal value : 7
Nodes evaluated: 29


---
## Task 2: Steepest-Ascent Hill Climbing

### 2A — Implementation

Implement `steepest_hill_climbing(initial_state, get_neighbors, evaluate)` that:
- Evaluates **ALL** neighbours before moving.
- Always moves to the neighbour with the **highest** evaluation value.
- Stops when no neighbour improves the current state (local optimum reached).
- Returns: `(best_state, best_value, steps_taken)`.

In [12]:
def steepest_hill_climbing(initial_state, get_neighbors, evaluate):
    """
    Steepest-Ascent Hill Climbing.

    Args:
        initial_state : starting state (any type your evaluate/get_neighbors expect)
        get_neighbors : function that takes a state and returns a list of neighbour states
        evaluate      : function that takes a state and returns a numeric score (higher = better)

    Returns:
        (best_state, best_value, steps_taken)
    """
    curr_state,  curr_value  = initial_state, evaluate(initial_state)
    flag = True
    step_count = 0
    while flag:
        flag = False
        for neighbor in get_neighbors(curr_state):
            if evaluate(neighbor) > curr_value:
                curr_state, curr_value = neighbor, evaluate(neighbor)
                flag = True
        if flag:
            step_count += 1
            
    return(curr_state, curr_value, step_count)
        
    
    

### 2B — Comparison

Run your function from starting states **0, 5, and 12** using the functions below.  
Note: neighbours of `x` are `x−1` and `x+1` — negative values are valid and will simply score lower, no bounds needed.  
Print a formatted table and add a short note explaining why different starting states reach the same optimum but may take a **different number of steps**.


In [13]:
def evaluate(x):      return -(x - 7) ** 2 + 49   # peak at x=7, value=49
def get_neighbors(x): return [x - 1, x + 1]

starting_states = [0, 5, 12]

# YOUR CODE HERE
# Run steepest_hill_climbing for each starting state
# Print a formatted table: Starting State | Best State | Best Value | Steps Taken

print(f"{'Start':>10} {'Best State':>12} {'Best Value':>12} {'Steps':>8}")
print("-" * 46)
for start in starting_states:
    state, value, steps = steepest_hill_climbing(start, get_neighbors, evaluate)
    print(f"Start:    {start}, State:   {state}, Value:   {value}, Steps:     {steps}")
    #print(f"{f"Best State: {state}":>8} {f"Best Value: {value}":>8} {f"Steps: {steps}":>8} ")
print("-" * 46)
    

     Start   Best State   Best Value    Steps
----------------------------------------------
Start:    0, State:   7, Value:   49, Steps:     7
Start:    5, State:   7, Value:   49, Steps:     2
Start:    12, State:   7, Value:   49, Steps:     5
----------------------------------------------


**Note:**  
_Write your observation here — since this function has a single peak at x=7, all starting states converge to the same optimum. They differ only in the number of steps (distance from 7).

---
## Task 3: Exam Hall Seating (N-Queens)

FAST-NU needs to seat **N students** in an **N×N exam hall**. Each student is from a different class. To prevent copying, no two students may sit in the same **row, column, or diagonal**.

**State representation:** a list of N column positions, one per row.  
One student per row is always guaranteed — the algorithm only needs to fix column and diagonal conflicts.

```python
state = [1, 3, 0, 2]   # N=4
# Row 0: student sits at column 1
# Row 1: student sits at column 3
# Row 2: student sits at column 0
# Row 3: student sits at column 2
```

This task builds directly on `steepest_hill_climbing` from Task 2A.

### 3A — conflicts(state)

Count the total number of conflicting student pairs.  
Two students conflict if they share the same **column** or **diagonal**.  
Return `0` for a valid seating arrangement.

```
# Conflict conditions for students at rows i and j (i != j):
# Same column  : state[i] == state[j]
# Same diagonal: abs(state[i] - state[j]) == abs(i - j)
```

In [14]:
def conflicts(state):
    """
    Count total conflicting pairs in the seating state.

    Args:
        state : list of N column positions (one per row)

    Returns:
        int: number of conflicting pairs (0 = valid seating)
    """
    count = 0
    curr = state[0]
    for i in range(len(state)):
        for j in range(i + 1, len(state)):
            # same column check
            if state[i] == state[j]:
                count += 1
            # same diagonal check
            if abs(state[i] - state[j]) == abs(i - j):
                count += 1

    return count


# Quick test — this state has 0 conflicts for N=4
test_state = [1, 3, 0, 2]
print(f"Conflicts in {test_state}: {conflicts(test_state)}")  # Expected: 0

Conflicts in [1, 3, 0, 2]: 0


### 3B — get_neighbors_hall(state)

Return all states reachable by moving exactly **one student one column left or right**.  
Skip any move that would take a student outside the valid column range **0 to N−1**.  
Do not modify the original state — create a new list for each neighbour.

> **Important:** This function is named `get_neighbors_hall` to avoid conflicting with the simple `get_neighbors` from Task 2B. You will pass `get_neighbors_hall` as the `get_neighbors` argument to `steepest_hill_climbing` in Task 3C.

In [15]:
def get_neighbors_hall(state):
    """
    Generate all neighbours by moving one student one column left or right.

    Args:
        state : list of N column positions

    Returns:
        list of neighbour states
    """
    n = len(state)
    neighbors = []
    for i in range(n):
        # move left
        if state[i] - 1 in range(0, n):
            neighbors.append(
                [state[j] - 1 if j == i else state[j] for j in range(n)]
            )

        # move right
        if state[i] + 1 in range(0, n):
            neighbors.append(
                [state[j] + 1 if j == i else state[j] for j in range(n)]
            )
    return neighbors

# Quick test
print(f"Number of neighbours for N=4: {len(get_neighbors_hall([1, 3, 0, 2]))}")
for neighbor in get_neighbors_hall([1, 3, 0, 2]):
    print(neighbor)
# Expected: up to 8 (2 moves per row), fewer if students are at edges

Number of neighbours for N=4: 6
[0, 3, 0, 2]
[2, 3, 0, 2]
[1, 2, 0, 2]
[1, 3, 1, 2]
[1, 3, 0, 1]
[1, 3, 0, 3]


### 3C — random_restart_hc(n, max_restarts=20)

Use `random.sample(range(n), n)` to generate a random initial state — this gives each student a unique starting column.  
Call `steepest_hill_climbing` from Task 2A, passing:
- `get_neighbors = get_neighbors_hall` (from Task 3B)
- `evaluate = lambda s: -conflicts(s)` so that fewer conflicts = higher value.

Repeat until a valid seating is found or `max_restarts` is reached.

> **Note:** Your `steepest_hill_climbing` must work with **list** states (not just numbers). It should already work if it only calls `evaluate(state)` and `get_neighbors(state)` without assuming the state type.

> **Note:** `random.sample` produces a permutation (unique columns), but `get_neighbors_hall` can break this uniqueness during hill climbing — that is expected. The `conflicts` function will penalise duplicate columns.

Returns:
- `(state, restarts_used)` if a solution is found (conflicts == 0)
- `(None, max_restarts)` if no solution found within the limit

In [16]:
import random

def random_restart_hc(n, max_restarts=20):
    """
    Hill Climbing with random restarts for the exam hall seating problem.

    Args:
        n            : number of students / hall size
        max_restarts : maximum number of random restarts allowed

    Returns:
        (solution_state, restarts_used) if found
        (None, max_restarts)            if not found within limit
    """
    for i in range(max_restarts):
        state, value, steps = steepest_hill_climbing(
            initial_state= random.sample(range(n), n),
            get_neighbors= get_neighbors_hall,
            evaluate= lambda s: -conflicts(s)
        )
        if value == 0:
            return (state, i)

    return (None, max_restarts)
            

### 3D — print_hall(state)

Print the seating grid. Use `[S]` for a seated student and `[ ]` for an empty seat.

Expected output for `[1, 3, 0, 2]`:
```
     C0   C1   C2   C3
R0  [ ]  [S]  [ ]  [ ]
R1  [ ]  [ ]  [ ]  [S]
R2  [S]  [ ]  [ ]  [ ]
R3  [ ]  [ ]  [S]  [ ]
```

In [22]:
def print_hall(state):
    """
    Print the exam hall seating grid.

    Args:
        state : list of N column positions
    """
    n = len(state)
    print('    ', end="")
    for i in range(n):
        print(f" C{i} ", end="")
    print('\n')
    for i in range(n):
        print(f" R{i}", end="")
        for j in range(n):
            value = ' '
            if j == state[i]:
                value = 'S'
            print(f" [{value}]", end="")
        print('\n')

# Quick test
print_hall([1, 3, 0, 2])

     C0  C1  C2  C3 

 R0 [ ] [S] [ ] [ ]

 R1 [ ] [ ] [ ] [S]

 R2 [S] [ ] [ ] [ ]

 R3 [ ] [ ] [S] [ ]



### 3E — Run and report

Run for **N=6** and **N=8**. For each, print:
- The seating grid using `print_hall`.
- Total conflicts in the final state (should be 0).
- Number of restarts needed.

In [27]:
for n in [6, 8]:
    print(f"\n{'='*40}")
    print(f" N = {n}")
    print(f"{'='*40}")

    solution, restarts = random_restart_hc(n)

    if solution is not None:
        print_hall(solution)
        print(f"Conflicts    : {conflicts(solution)}")
        print(f"Restarts used: {restarts}")
    else:
        print(f"No solution found within {restarts} restarts.")


 N = 6
     C0  C1  C2  C3  C4  C5 

 R0 [ ] [ ] [ ] [S] [ ] [ ]

 R1 [S] [ ] [ ] [ ] [ ] [ ]

 R2 [ ] [ ] [ ] [ ] [S] [ ]

 R3 [ ] [S] [ ] [ ] [ ] [ ]

 R4 [ ] [ ] [ ] [ ] [ ] [S]

 R5 [ ] [ ] [S] [ ] [ ] [ ]

Conflicts    : 0
Restarts used: 3

 N = 8
No solution found within 20 restarts.


---
## Submission Checklist

Before submitting, make sure:
- [x] All cells are executed and outputs are visible.
- [x] `nodes_evaluated` is printed in Task 1B.
- [x] Task 2B table is printed and the note is filled in.
- [x] Task 3E shows valid grids for both N=6 and N=8 with 0 conflicts.
- [x] File is renamed to your **roll number** (e.g. `21L-1234.ipynb`).